## Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import datetime
import einops
from inductive_bias.MLP_metrics import measure_metrics
import copy
from inductive_bias.local_cache import CacheContext
from tyche.math import gaussint_ln_noncentral_erf
from tyche.math import gaussint_ln_riemann
from tyche.utils import weighted_logsumexp
from torch import nn
import torch as t
from typing import List
from jaxtyping import Float, Int
from inductive_bias.MLP_init import *
from inductive_bias.MLP_metrics import run_train_and_estimator
from inductive_bias.inductive_bias_viz import generate_heatmaps
import itertools
import plotly.graph_objs as go  # Import the graph objects from Plotly
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from tqdm.notebook import tqdm
import pandas as pd
from tyche.estimator import VolumeConfig, VolumeEstimator

In [3]:
device = t.device("cuda:7" if t.cuda.is_available() else "cpu")
t.set_default_device(device)

## Evaluating hyperparameter search

In [10]:
WEIGHT_MODES = ["uniform", "xavier_uniform", "normal", "xavier_normal", "constant"]

In [ ]:
path = "/home/louis/tyche/scripts/shared_database_3.parquet"
df_new = pd.read_parquet(path)
WEIGHT_MODES = ["none"]
plots = generate_heatmaps(df=df_new, weight_mode="none", cmap="coolwarm")

In [ ]:
path = "/home/louis/tyche/scripts/shared_database_10.parquet"
df_old = pd.read_parquet(path)
WEIGHT_MODES = ["none"]
plots = generate_heatmaps(df=df_old, weight_mode="none", cmap="coolwarm")

## training attempt

In [9]:
params = MLPConfig(
    activation=t.nn.ReLU(),
    N=53,
    embed_dimension=36,
    linear_dimension=48,
    intermediate="pure",
    embedding_tied=False,
    unembedding_tied=False,
    bias_unembed=True,
    bias_layer=True,
    num_additional_layers=5,
    dimensions=(48),
    W_amplitude=1,
    weight_mode="none",
    device=device,
    b_amplitude=0.0,
    train_data_size=1600,
    training_epochs=10000,
    eval_interval=1000,
    weight_decay=2e-4,
    save_model=False,
)
model = MLP_VARIANTS(params)
model.initialize_weights()

volume_cfg = VolumeConfig(
    model_type="mlp",
    model=model,
    n_samples=100,
    iters=15,
    cutoff=1e-2,
    cache_mode=None,
    chunking=False,
    reduction=None,
    tol=0.0351,
    tqdm=False,
    gaussint_fn=gaussint_ln_riemann,
    new_estimator=False,
)
results = run_train_and_estimator(
    custom_config=params, gpu_id=7, volume_config=volume_cfg
)

/home/louis/tyche/.venv/lib/python3.13/site-packages/torch/utils/_device.py:104: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)
0it [00:00, ?it/s]


In [ ]:
from inductive_bias.inductive_bias_viz import plot_indicator_table


plot_indicator_table(model=model, params=params, save=True)

## Investigate local similarity

In [ ]:
ACTS = [t.nn.ReLU(), t.nn.GELU(), t.nn.Tanh()]

result_acts = {}
models_acts = []
from tyche.MLP_metrics import run_train_and_estimator

for act in ACTS:
    params = MLPConfig(
        activation=act,
        N=53,
        embed_dimension=36,
        linear_dimension=48,
        intermediate="pure",
        embedding_tied=False,
        unembedding_tied=False,
        bias_unembed=False,
        num_additional_layers=2,
        dimensions=(48),
        bias_layer=False,
        W_amplitude=np.sqrt(10) ** -0.5,
        weight_mode="xavier_uniform",
        device=device,
        b_amplitude=0.0,
        train_data_size=53**2,
        training_epochs=0,
        seed=1,
    )

    model = MLP_VARIANTS(params)
    model.initialize_weights()
    break
    models_acts.append(model)

    with CacheContext(f".cache/run_{act.__class__.__name__}"):
        result_dict = run_train_and_estimator(custom_config=params, gpu_id=7)[0]

    # result_dict = run_train_and_estimator(custom_config=params, gpu_id=7)[0]

    result_acts[f"{act}"] = result_dict

In [ ]:
from tyche.volume import VolumeResult

# Register it as a safe class with PyTorch
import torch.serialization

torch.serialization.add_safe_globals([VolumeResult])
cache_dict = {}
for act in ACTS:
    folder_dir = f".cache/run_{act.__class__.__name__}"

    subfolders = [f.path for f in os.scandir(folder_dir)]
    print(act)
    tensors = [
        t.load(subfolder, weights_only=False)
        for subfolder in subfolders
        if subfolder.endswith(".pt")
    ]

    cache_dict[f"{act}"] = tensors

In [ ]:
cache_dict["GELU(approximate='none')"][0]["input"]

In [ ]:
cache_dict["Tanh()"][0]

In [ ]:
for k, v in cache_dict.items():
    diff = v - cache_dict["ReLU()"]
    print(diff)

In [ ]:
cache_dict["ReLU()"][:10]

In [ ]:
cache_dict["Tanh()"][:10]

In [155]:
volume_estimates = [result_acts[act]["volume_estimates"] for act in result_acts.keys()]

In [139]:
for v in volume_estimates:
    diff = v - volume_estimates[0]
    print((diff**2).sum())

In [ ]:
models_acts[0]

In [141]:
model_vec = t.nn.utils.parameters_to_vector(model.parameters())
new_vec = model_vec + 0
new_model = copy.deepcopy(model)
t.nn.utils.vector_to_parameters(new_vec, new_model.parameters())

In [157]:
d_model = model_vec.shape[0]

In [156]:
def random_direction_model(
    model: MLP_VARIANTS,
    direction: Float[t.Tensor, "d_params"],
    mult: float = 1.0,
    dataset: Optional[Int[t.Tensor, "train_set_size 2"]] = None,
):

    if dataset is None:
        dataset = t.tensor(
            list(itertools.product(range(model.mlp_config.N), repeat=2)),
            device=model.mlp_config.device,
        )

    model_vec = t.nn.utils.parameters_to_vector(model.parameters())
    new_vec = model_vec + direction * mult
    new_model = copy.deepcopy(model)
    t.nn.utils.vector_to_parameters(new_vec, new_model.parameters())

    logits = model(dataset)
    new_logits = new_model(dataset)

    probs = t.nn.functional.softmax(logits, dim=-1)
    logprobs_new = t.nn.functional.log_softmax(new_logits, dim=-1)

    kl_div = (
        t.nn.functional.kl_div(logprobs_new, probs, reduction="none").sum(dim=-1).mean()
    )

    return kl_div

In [144]:
direction = t.randn(d_model, device=device)
direction = direction / direction.norm()
kl_divs = {f"{model.mlp_config.activation}": [] for model in models_acts}

In [ ]:
direction = t.randn(d_model, device=device)
direction.sum()

In [158]:
# different random_seed
direction = t.randn(d_model, device=device)
direction = direction / direction.norm()
kl_divs = {f"{model.mlp_config.activation}": [] for model in models_acts}
for model in models_acts:
    for mult in np.arange(0, 100, 0.5):
        kl_div = random_direction_model(model, direction, mult=mult)
        kl_divs[f"{model.mlp_config.activation}"].append(kl_div.item())

In [ ]:
print(kl_divs["ReLU()"][-1])

In [160]:
kl_nats = 1e-2

# Convert to bits
kl_bits = kl_nats * np.log2(np.e)

In [ ]:
for k, v in kl_divs.items():
    plt.plot(v[:50], label=k)
plt.axhline(y=kl_nats, color="red", linestyle="--", label="y=1e-2")
plt.xlabel("Mult")
plt.ylabel("KL Divergence")
plt.legend()
plt.title("KL Divergence for different activations")
plt.show()

## Complexity over time

In [6]:
path = "/home/louis/tyche/scripts/shared_database_True_2809.parquet"
df = pd.read_parquet(path)
epoch_list = df["epoch"].unique()

In [53]:
df_epoch = df[df["epoch"] == 5000].copy()

In [60]:
mask = (
    (df_epoch["activation"] == "ReLU")
    & (df_epoch["num_additional_layers"] == 5)
    & (df_epoch["intermediate"] == "pure")
)

In [66]:
df_specific = df_epoch[mask]

hashable_columns = []
for col in df_specific.columns:
    try:
        num_unique = df_specific[col].nunique()
        if num_unique > 1:
            hashable_columns.append(col)
    except TypeError:  # This will catch the unhashable type error
        pass  # Skip columns with unhashable types

filtered_df = df_specific[hashable_columns]

In [ ]:
filtered_df[filtered_df["W_amplitude"] == 1.0]

In [ ]:
df_epoch = df[df["epoch"] == 10000].copy()
generate_heatmaps(
    df=df_epoch,
    weight_mode="none",
    stat_type="mean",
    value_stat="mean",
    figsize=(15, 10),
    cmap="coolwarm",
    value_column="test_loss",
)

In [ ]:
figs_dict = {
    "figs_volume": [],
    "figs_test_loss": [],
    "figs_test_accuracy": [],
    "figs_train_loss": [],
    "figs_train_accuracy": [],
}
for epoch in epoch_list:
    mask = df["epoch"] == epoch
    df_epoch = df[mask]
    fig_volume = generate_heatmaps(
        df=df_epoch,
        weight_mode="none",
        stat_type="mean",
        value_stat="mean",
        figsize=(15, 10),
        cmap="coolwarm",
    )
    figs_dict["figs_volume"].append(fig_volume)
    fig_test_loss = generate_heatmaps(
        df=df_epoch,
        weight_mode="none",
        stat_type="mean",
        value_stat="mean",
        figsize=(15, 10),
        cmap="coolwarm",
        value_column="test_loss",
    )
    figs_dict["figs_test_loss"].append(fig_test_loss)
    fig_test_accuracy = generate_heatmaps(
        df=df_epoch,
        weight_mode="none",
        stat_type="mean",
        value_stat="mean",
        figsize=(15, 10),
        cmap="coolwarm",
        value_column="test_accuracy",
    )
    figs_dict["figs_test_accuracy"].append(fig_test_accuracy)
    fig_train_loss = generate_heatmaps(
        df=df_epoch,
        weight_mode="none",
        stat_type="mean",
        value_stat="mean",
        figsize=(15, 10),
        cmap="coolwarm",
        value_column="train_loss",
    )
    figs_dict["figs_train_loss"].append(fig_train_loss)
    fig_train_accuracy = generate_heatmaps(
        df=df_epoch,
        weight_mode="none",
        stat_type="mean",
        value_stat="mean",
        figsize=(15, 10),
        cmap="coolwarm",
        value_column="train_accuracy",
    )
    figs_dict["figs_train_accuracy"].append(fig_train_accuracy)

In [ ]:
figs_dict

In [31]:
epoch_list = [epoch.item() for epoch in df["epoch"].unique()]

In [ ]:
from inductive_bias.inductive_bias_viz import export_figure_dict_to_html


interactive_figure = export_figure_dict_to_html(
    figure_dict=figs_dict, epoch_list=epoch_list
)

In [20]:
df_test = df[df["epoch"] == 0].copy()

In [ ]:
df_test["activation"].unique()

In [ ]:
# ploy df_test
generate_heatmaps(
    df=df_test,
    weight_mode="none",
    stat_type="mean",
    volume_stat="mean",
    cmap="coolwarm",
    title="epoch 0",
)

## Gaussian integral


In [6]:
from tyche.volume import VolumeResult

# Register it as a safe class with PyTorch
import torch.serialization

torch.serialization.add_safe_globals([VolumeResult])
ACTS = [t.nn.ReLU(), t.nn.GELU(), t.nn.Tanh()]
cache_dict = {}
for act in ACTS:
    folder_dir = f".cache/run_{act.__class__.__name__}"

    subfolders = [f.path for f in os.scandir(folder_dir)]

    tensors = [
        t.load(subfolder, weights_only=False)
        for subfolder in subfolders
        if subfolder.endswith(".pt")
    ]

    cache_dict[f"{act.__class__.__name__}"] = tensors

input = cache_dict["ReLU"][0]["input"]["kwargs"]
input_test = copy.deepcopy(input)

## Testing new integral with Pythia

In [19]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

from tyche import VolumeConfig, VolumeEstimator


# Load any CausalLM model, tokenizer, and dataset
model = AutoModelForCausalLM.from_pretrained("EleutherAI/pythia-14m")
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-14m")
tokenizer.pad_token_id = 1  # pythia-specific
tokenizer.eos_token_id = 0  # pythia-specific
dataset = load_dataset(
    "EleutherAI/lambada_openai", name="en", split="test", trust_remote_code=True
)

# Configure the estimator
cfg = VolumeConfig(
    model=model,
    tokenizer=tokenizer,
    dataset=dataset,
    text_key="text",  # must match dataset field
    n_samples=3,  # number of MC samples
    cutoff=1e-2,  # KL-divergence cutoff (nats)
    max_seq_len=2048,  # max sequence length for tokenizer or chunk_and_tokenize
    val_size=10,  # number of dataset sequences to use. default (None) uses all.
    cache_mode=None,  # see below
    chunking=False,  # whether to use chunk_and_tokenize
    gaussint_fn=gaussint_ln_riemann,
    new_estimator=True,  # use the new estimator
)
estimator = VolumeEstimator.from_config(cfg)


# Run the estimator

tokens.shape=torch.Size([10, 224])


In [20]:
result = estimator.run()

  0%|          | 0/3 [00:00<?, ?it/s]

/home/louis/tyche/.venv/lib/python3.13/site-packages/torch/utils/_device.py:104: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)
100%|██████████| 3/3 [00:02<00:00,  1.09it/s]


In [21]:
result

VolumeResult(estimates=tensor([[-1.4030e+08],
        [-1.4030e+08],
        [-1.3949e+08]], device='cuda:0'), props=tensor([1., 1., 1.], device='cuda:0'), mults=tensor([[0.5312],
        [0.5312],
        [0.5625]], device='cuda:0'), deltas=tensor([0.0103, 0.0102, 0.0108], device='cuda:0'), gaussint=tensor([[-15932714.],
        [-15932714.],
        [-15128628.]], device='cuda:0'))